In [26]:
import sys
import json
import warnings
from pathlib import Path
from datetime import timedelta
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve().parent
load_dotenv(REPO_ROOT / '.env')          # 
sys.path.insert(0, str(REPO_ROOT))       # audio_engine

from lib import brand, animation, data_quiver, data_massive, backtest
from audio_engine import Cue, render_track, ticks_every

warnings.filterwarnings('ignore')
brand.apply_theme()                      #
print('Imports OK')

Imports OK


In [27]:

HOLD_DAYS       = 60      # calendar days
MIN_TRADES_RANK = 5       # minimum trades to appear on politician ranking chart
PRICE_END       = '2026-07-02'   # last complete trading day at time of analysis


OUT_DIR = Path('../output/1-buy-the-day-they-bought')
OUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR = Path('cache/audio')          # working WAVs (gitignored)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output directory: {OUT_DIR.resolve()}')

Output directory: G:\congress-trades-investigation\output\1-buy-the-day-they-bought


In [28]:
# Full disclosed Congress trading history (Quiver bulk endpoint, paginated)
raw_congress = data_quiver.fetch_congress_trades()
print(f'{len(raw_congress):,} records | columns: {list(raw_congress.columns)}')
raw_congress.head(3)

114,217 records | columns: ['Ticker', 'TickerType', 'Company', 'TransactionDate', 'Transaction', 'Amount', 'Status', 'Subholding', 'Description', 'Representative', 'BioGuideID', 'ReportDate', 'Party', 'District', 'House', 'Comments', 'Quiver_Upload_Time', 'excess_return', 'State', 'last_modified']


,Ticker,TickerType,Company,TransactionDate,Transaction,Amount,Status,Subholding,Description,Representative,BioGuideID,ReportDate,Party,District,House,Comments,Quiver_Upload_Time,excess_return,State,last_modified
0,BLK,ST,None,2026-07-07,Sale,1001.0,None,None,None,John Mcguire,M001239,2026-07-08,Republican,5.0,Representatives,None,None,0.465932405751667,None,2026-07-09
1,FLL,ST,None,2026-07-01,Sale,1001.0,None,None,None,Susie Lee,L000590,2026-07-08,Democratic,3.0,Representatives,None,None,-5.93287205507722,None,2026-07-09
2,NVDA,Stock,None,2026-06-30,Sale (Partial),15001.0,None,None,None,Sheldon Whitehouse,W000802,2026-07-08,Democratic,None,Senate,None,None,0.512802901570477,None,2026-07-09


In [29]:
# Purchases only, valid tickers + dates, deduplicated (DJT excluded — his own company)
congress_df = data_quiver.clean_congress_buys(raw_congress)
print(f'{len(congress_df):,} buy trades | {congress_df["Ticker"].nunique()} tickers '
      f'| {congress_df["Representative"].nunique()} reps')
print(f'{congress_df["TransactionDate"].min().date()} → {congress_df["TransactionDate"].max().date()}')
congress_df.head(3)

47,975 buy trades | 3777 tickers | 249 reps
2012-06-06 → 2026-06-30


,Ticker,TickerType,Company,TransactionDate,Transaction,Amount,Status,Subholding,Description,Representative,BioGuideID,ReportDate,Party,District,House,Comments,Quiver_Upload_Time,excess_return,State,last_modified
0,KSU,None,None,2012-06-06,Purchase,1001.0,None,None,None,Alan S. Lowenthal,L000579,2014-05-15,Democratic,47.0,Representatives,None,None,89.2382297931922,None,2023-11-16
1,CAT,None,None,2012-07-26,Purchase,1001.0,None,None,None,Tammy Duckworth,D000622,2014-07-08,Democratic,None,Senate,None,None,598.674780334488,None,2023-11-16
2,DD,Stock,None,2012-09-13,Purchase,1001.0,None,None,None,Thomas R. Carper,C000174,2015-05-13,Democratic,None,Senate,None,None,-441.854724586666,None,2023-08-28


In [30]:

fetch_start = (congress_df['TransactionDate'].min() - timedelta(days=5)).strftime('%Y-%m-%d')
print(f'Price window: {fetch_start} → {PRICE_END}')

Price window: 2012-06-01 → 2026-07-02


In [31]:
tickers = ['SPY'] + sorted(congress_df['Ticker'].unique().tolist())
price_cache = data_massive.load_price_cache(tickers, fetch_start, PRICE_END)
spy_prices = price_cache['SPY']
print(f'SPY: {len(spy_prices)} trading days  ({spy_prices.index[0].date()} → {spy_prices.index[-1].date()})')

Price cache: 4801 tickers cached | 1 to fetch
Done: 3536 tickers with data | 242 empty (delisted/bad symbol)
SPY: 3541 trading days  (2012-06-01 → 2026-07-02)


In [32]:
# Backtest v1  perfect information: enter the day they traded, hold 60 days
v1_results = backtest.run_backtest(congress_df, price_cache, spy_prices,
                                   hold_days=HOLD_DAYS, entry='transaction')
v1_results.head(3)

44,294 trades complete | 3681 skipped
Avg return: 1.94% | SPY: 2.33% | Excess: -0.39%
Win rate: 56.07% | Beat SPY: 47.26%


,Ticker,Representative,Party,House,TransactionDate,ReportDate,EntryDate,ExitDate,EntryPrice,ExitPrice,TradeReturn,SPYReturn,ExcessReturn,Open
0,KSU,Alan S. Lowenthal,Democratic,Representatives,2012-06-06,2014-05-15,2012-06-06,2012-08-05,66.0500,73.3900,0.111128,0.057968,0.053160,False
1,CAT,Tammy Duckworth,Democratic,Senate,2012-07-26,2014-07-08,2012-07-26,2012-09-24,83.3000,90.8700,0.090876,0.069582,0.021294,False
2,DD,Thomas R. Carper,Democratic,Senate,2012-09-13,2015-05-13,2012-09-13,2012-11-12,192.5397,162.4143,-0.156463,-0.056791,-0.099672,False


In [33]:
v1_curve = backtest.equity_curve(v1_results, price_cache, spy_prices)
pol_stats = backtest.politician_stats(v1_results, min_trades=MIN_TRADES_RANK)

top5 = v1_results.nlargest(5, 'TradeReturn')
bot5 = v1_results.nsmallest(5, 'TradeReturn')
best = top5.iloc[0]

summary = backtest.summarize(v1_results, v1_curve)
summary.update({
    'engine': 'v1_transaction_date_entry',
    'hold_days': HOLD_DAYS,
    'portfolio_model': 'equal-weight daily rebalance across open positions',
    'top_politician': str(pol_stats.iloc[0]['Representative']),
    'top_politician_return': float(pol_stats.iloc[0]['AvgTradeReturn']),
    'best_trade': {   # the single best trade — called out in the script
        'representative': str(best['Representative']),
        'ticker': str(best['Ticker']),
        'date': str(best['TransactionDate'].date()),
        'return_60d': float(best['TradeReturn']),
    },
})

v1_results.to_csv(OUT_DIR / 'trade_results.csv', index=False)
top5.to_csv(OUT_DIR / 'top5_trades.csv', index=False)
bot5.to_csv(OUT_DIR / 'bottom5_trades.csv', index=False)
pol_stats.to_csv(OUT_DIR / 'politician_ranking.csv', index=False)
v1_curve.to_csv(OUT_DIR / 'equity_curve_data.csv', index=False)
with open(OUT_DIR / 'summary_stats.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"${backtest.INITIAL_CAPITAL:,} → portfolio ${summary['final_portfolio']:,.0f} "
      f"| SPY ${summary['final_spy']:,.0f}")
print(f"Top politician: {summary['top_politician']} ({summary['top_politician_return']:.2%} avg)")
print(f"Best single trade: {best['Representative']} — {best['Ticker']} "
      f"{best['TransactionDate'].date()} → {best['TradeReturn']:.1%} in {HOLD_DAYS} days")

$10,000 → portfolio $39,378 | SPY $56,586
Top politician: John Fetterman (38.17% avg)
Best single trade: Michael T. McCaul — SRM 2025-04-23 → 1954.5% in 60 days


In [34]:

DRAW_S, HOLD_S = 12.0, 2.0
wav = render_track(
    ticks_every('tick', 1.0, DRAW_S - 1.0, 2.0, gain_db=-8)
    + [Cue('line_swell', DRAW_S - 1.8), Cue('data_blip', DRAW_S + 0.4, gain_db=-4)],
    duration_s=DRAW_S + HOLD_S,
    out_path=AUDIO_DIR / 'd1_equity.wav',
)

animation.animate_lines(
    series=[
        {'label': 'Congress Copy Strategy', 'short': 'Congress', 'dates': v1_curve['Date'].values,
         'values': v1_curve['CumPortfolio'].values, 'color': brand.GREEN},
        {'label': 'S&P 500 Buy & Hold', 'short': 'SPY', 'dates': v1_curve['Date'].values,
         'values': v1_curve['CumSPY'].values, 'color': brand.OFF_WHITE, 'alpha': 0.9},
    ],
    title='COPY EVERY CONGRESS BUY - PERFECT INFO (60-DAY HOLD)',
    output_path=OUT_DIR / 'equity_curve.mp4',
    draw_seconds=DRAW_S, hold_seconds=HOLD_S,
    ref_line=backtest.INITIAL_CAPITAL,
    audio_wav=wav,
)

Rendering equity_curve.mp4  (840 frames @ 60 fps)...
Saved equity_curve.mp4  (1.0 MB)


840

In [35]:

import pandas as pd

def trade_label(row):
    last = str(row['Representative']).split()[-1]
    return f"{row['Ticker']}\n{last}\n{pd.Timestamp(row['TransactionDate']):%b %Y}"

combined = pd.concat([top5, bot5])
GROW_S, HOLD_S = 1.5, 2.5
wav = render_track(
    [Cue('bar_grow', 0.1), Cue('data_blip', GROW_S, gain_db=-4),
     Cue('error_buzz', GROW_S + 0.5, gain_db=-10)],   # soft buzz as the red half lands
    duration_s=GROW_S + HOLD_S,
    out_path=AUDIO_DIR / 'd1_topbottom.wav',
)

animation.animate_bars(
    labels=[trade_label(r) for _, r in combined.iterrows()],
    values=combined['TradeReturn'].tolist(),
    colors=[brand.GREEN if r >= 0 else brand.RED for r in combined['TradeReturn']],
    title='TOP 5 & BOTTOM 5 TRADES - PERFECT INFO ENTRY',
    output_path=OUT_DIR / 'top5_bottom5_trades.mp4',
    grow_seconds=GROW_S, hold_seconds=HOLD_S,
    divider_after=5, group_labels=('TOP 5', 'BOTTOM 5'),
    audio_wav=wav,
)

Rendering top5_bottom5_trades.mp4  (240 frames @ 60 fps)...
Saved top5_bottom5_trades.mp4  (0.2 MB)


240

In [36]:

TOP_N = 15
top_pols = pol_stats.head(TOP_N).sort_values('AvgTradeReturn').reset_index(drop=True)

def pol_label(name):
    last = str(name).split()[-1]
    return f'{last} *' if last.lower() == 'pelosi' else last

GROW_S, HOLD_S = 1.5, 2.5
wav = render_track(
    [Cue('bar_grow', 0.1)] + ticks_every('tick', GROW_S, GROW_S + 0.7, 0.05, gain_db=-10)
    + [Cue('data_blip', GROW_S + 1.0, gain_db=-4)],
    duration_s=GROW_S + HOLD_S,
    out_path=AUDIO_DIR / 'd1_ranking.wav',
)

animation.animate_bars(
    labels=[pol_label(n) for n in top_pols['Representative']],
    values=top_pols['AvgTradeReturn'].tolist(),
    colors=[brand.GREEN] * len(top_pols),
    title=f'TOP {TOP_N} POLITICIANS - AVG TRADE RETURN (PERFECT INFO, >={MIN_TRADES_RANK} TRADES)',
    output_path=OUT_DIR / 'politician_ranking.mp4',
    horizontal=True,
    grow_seconds=GROW_S, hold_seconds=HOLD_S,
    annotations=[f' {n}t' for n in top_pols['NumTrades']],
    audio_wav=wav,
)

pelosi = pol_stats[pol_stats['Representative'].str.contains('Pelosi', case=False)]
if not pelosi.empty:
    p = pelosi.iloc[0]
    print(f"Pelosi: {p['NumTrades']} trades | avg {p['AvgTradeReturn']:.2%} "
          f"| beat SPY {p['BeatSPYRate']:.0%} of the time")

Rendering politician_ranking.mp4  (240 frames @ 60 fps)...
Saved politician_ranking.mp4  (0.2 MB)
Pelosi: 98 trades | avg 0.19% | beat SPY 50% of the time
